# 📥 Notebook 01 — Data Sourcing & Initial Audit
### HR Analytics: Job Change of Data Scientists
**Objective:** Load raw dataset, perform structural audit, document quality issues.
> Place `aug_train.csv` in `../data/raw/` before running.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import os, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'primary': '#1B3A6B', 'accent': '#E84855', 'safe': '#2ECC71'}

RAW_PATH = '../data/raw/aug_train.csv'
print(f'File exists: {os.path.exists(RAW_PATH)}')

## 1. Load & Shape

In [ ]:
df = pd.read_csv(RAW_PATH)
print(f'Shape: {df.shape}  |  Memory: {df.memory_usage(deep=True).sum()/1e6:.2f} MB')
df.head(10)

## 2. Schema & Missing Value Audit

In [ ]:
audit = pd.DataFrame({
    'dtype':    df.dtypes,
    'non_null': df.notnull().sum(),
    'null':     df.isnull().sum(),
    'null_pct': (df.isnull().mean() * 100).round(2),
    'unique':   df.nunique()
})
print(audit.to_string())

## 3. Missing Value Visual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)
axes[0].barh(missing.index, missing.values, color=COLORS['accent'])
for i, v in enumerate(missing.values):
    axes[0].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
axes[0].set_xlabel('Missing %'); axes[0].set_title('Missing Values by Column')
msno.matrix(df, ax=axes[1], color=(0.106, 0.227, 0.420), sparkline=False)
axes[1].set_title('Missingness Pattern Matrix')
plt.tight_layout()
plt.savefig('../data/raw/01_missing_audit.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Target Distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
tc = df['target'].value_counts()
labels = ['Not Switching', 'Switching']
ax[0].bar(labels, tc.values, color=[COLORS['safe'], COLORS['accent']], edgecolor='white')
for i, v in enumerate(tc.values):
    ax[0].text(i, v+150, f'{v:,} ({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
ax[0].set_title('Target Class Counts')
ax[1].pie(tc.values, labels=labels, colors=[COLORS['safe'], COLORS['accent']],
          autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
ax[1].set_title('Target Split')
plt.suptitle('⚠ Imbalanced Dataset — 24.9% Positive Class', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/raw/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Imbalance ratio: {tc[0]/tc[1]:.2f}:1  |  Baseline accuracy: {tc[0]/len(df)*100:.1f}%')

## 5. Categorical Feature Counts

In [ ]:
cat_cols = df.select_dtypes('object').columns.tolist()
fig, axes = plt.subplots(3, 3, figsize=(20, 15))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    vc = df[col].value_counts(dropna=False)
    axes[i].barh(vc.index.astype(str), vc.values, color=COLORS['primary'], alpha=0.85)
    axes[i].set_title(f'{col}  |  missing={df[col].isnull().mean()*100:.1f}%')
    axes[i].set_xlabel('Count')
for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Categorical Feature Distributions (Raw)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/raw/01_categorical_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Numeric Distributions

In [ ]:
num_cols = ['city_development_index', 'training_hours']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=40, color=COLORS['primary'], alpha=0.85, edgecolor='white')
    axes[i].axvline(df[col].median(), color=COLORS['accent'], ls='--', label=f'Median={df[col].median():.3f}')
    axes[i].axvline(df[col].mean(),   color=COLORS['safe'],   ls='-',  label=f'Mean={df[col].mean():.3f}')
    axes[i].set_title(f'{col}')
    axes[i].legend()
plt.suptitle('Numeric Feature Distributions', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/raw/01_numeric_dist.png', dpi=150, bbox_inches='tight')
plt.show()
df[num_cols].describe().round(3)

## 7. Audit Summary

In [ ]:
issues = [
    ('company_type',        '32.0%', 'Create Unknown category — likely unemployed/freelancer'),
    ('company_size',        '31.0%', 'Create Unknown category — correlates with company_type'),
    ('gender',              '23.5%', 'Mode impute + retain Unknown flag'),
    ('major_discipline',    '14.7%', 'Contextual impute based on education_level'),
    ('enrolled_university',  '2.0%', 'Mode impute with no_enrollment'),
    ('education_level',      '2.4%', 'Mode impute + ordinal encode'),
    ('experience',           '0.3%', 'Ordinal map: <1→0, 1-20, >20→21'),
    ('last_new_job',         '2.2%', 'Ordinal map: never→0, 1-4, >4→5'),
    ('training_hours',       '0.0%', 'Check outliers (max=336 hrs)'),
    ('city',                 '0.0%', 'Drop — use city_development_index proxy'),
]
print(f'{"Column":<25} {"Null%":<10} {"Treatment"}')
print('-'*80)
for col, pct, action in issues:
    print(f'  {col:<23} {pct:<10} {action}')
print('\n✅ Audit complete. Proceeding to 02_cleaning.ipynb')

---
## ✅ Next: `02_cleaning.ipynb` — ETL Pipeline